# Tutorial 3: Function Secret Sharing


## DPF

$$
f_{\alpha,\beta}(x)=\begin{cases}
\beta, & \text {if x=$\alpha$} \\
0, & \text {else}
\end{cases}
$$


In [1]:
# import the libraries
import torch
from crypto.primitives.function_secret_sharing.dpf import DPF
from crypto.tensor.RingTensor import RingTensor

num_of_keys = 10  # We need a few keys for a few function values, but of course we can generate many keys in advance.

# generate keys in offline phase
# set alpha and beta
alpha = RingTensor.convert_to_ring(torch.tensor(5))
beta = RingTensor.convert_to_ring(torch.tensor(1))

Key0, Key1 = DPF.gen(num_of_keys=num_of_keys, alpha=alpha, beta=beta)

# online phase
# generate some values what we need to evaluate
x = RingTensor.convert_to_ring(torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]))

# Party 0:
res_0 = DPF.eval(x=x, keys=Key0, party_id=0)

# Party 1:
res_1 = DPF.eval(x=x, keys=Key1, party_id=1)

# restore result
res = res_0 + res_1
print(res)

tensor([0, 0, 0, 0, 1, 0, 0, 0, 0, 0], device='cuda:0')


## DCF

$$
f_{\alpha,\beta}(x)=\begin{cases}
\beta, & \text {if x < $\alpha$} \\
0, & \text {else}
\end{cases}
$$


In [2]:
# import the libraries
import torch
from crypto.primitives.function_secret_sharing.dcf import DCF
from crypto.tensor.RingTensor import RingTensor

num_of_keys = 10  # We need a few keys for a few function values, but of course we can generate many keys in advance.

# generate keys in offline phase
# set alpha and beta
alpha = RingTensor.convert_to_ring(torch.tensor(5))
beta = RingTensor.convert_to_ring(torch.tensor(1))

Key0, Key1 = DCF.gen(num_of_keys=num_of_keys, alpha=alpha, beta=beta)

# online phase
# generate some values what we need to evaluate
x = RingTensor.convert_to_ring(torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]))

# Party 0:
res_0 = DCF.eval(x=x, keys=Key0, party_id=0)

# Party 1:
res_1 = DCF.eval(x=x, keys=Key1, party_id=1)

# restore result
res = res_0 + res_1
print(res)

tensor([1, 1, 1, 1, 0, 0, 0, 0, 0, 0], device='cuda:0')


## $DICF$
f_{p,q}(x)=\begin{cases}
1, & \text {if p$\leq$x $\leq$ q} \\
0, & \text {else}
\end{cases}
$$


In [3]:
# import the libraries
# import the libraries
import torch
from crypto.primitives.function_secret_sharing.dicf import DICF
from crypto.tensor.RingTensor import RingTensor
from config.base_configs import DEVICE

# generate key in offline phase
num_of_keys = 10
down_bound = torch.tensor([3]).to(DEVICE)
upper_bound = torch.tensor([7]).to(DEVICE)

Key0, Key1 = DICF.gen(num_of_keys=num_of_keys, down_bound=down_bound, upper_bound=upper_bound)

# evaluate x in online phase
# generate some values what we need to evaluate
x = RingTensor.convert_to_ring(torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]))
x_shift = x + Key0.r.reshape(x.shape) + Key1.r.reshape(x.shape)

# online phase
# Party 0:
res_0 = DICF.eval(x_shift=x_shift, keys=Key0, party_id=0, down_bound=down_bound, upper_bound=upper_bound)

# Party 1:
res_1 = DICF.eval(x_shift=x_shift, keys=Key1, party_id=1, down_bound=down_bound, upper_bound=upper_bound)

# restore result
res = res_0 + res_1
print(res)

tensor([0, 0, 1, 1, 1, 1, 1, 0, 0, 0], device='cuda:0')
